# SGAT-CLMDA
## Sparse Graph Attention Network with Contrastive Learning
## for Microbe-Disease Association Prediction

**Dataset**: HMDAD (450 associations, 39 diseases, 292 microbes)

### Notebook Contents
| Cell | Description |
|------|-------------|
| 1–7  | Setup, data, model, loss functions |
| 8    | Main 5-Fold CV (tracks τ per fold) |
| 9    | Leakage comparison (honest vs leaky protocol) |
| 10   | Ablation study (Full / No-CL / GCN-only / GAT-only) |
| 11   | Plot: Adaptive threshold τ evolution |
| 12   | Plot: Leakage protocol comparison |
| 13   | Plot: Ablation study results |
| 14   | Plot: ROC curves |
| 15   | Plot: PR curves |
| 16   | Plot: Training loss |
| 17   | Train final model on all data |
| 18   | Case study predictions (top-10 per disease) |
| 19   | Top-20 microbe contributors |
| 20   | Plot: Case study bar charts |
| 21   | Plot: Top contributors |
| 22   | Plot: Score distribution |
| 23   | Plot: UMAP embeddings |
| 24   | Summary dashboard |
| 25   | Save all outputs |

In [ ]:
# ── CELL 1: Install Dependencies ──────────────────────────
import torch, sys
print(f'PyTorch : {torch.__version__}')
print(f'CUDA    : {torch.version.cuda}')
TORCH = torch.__version__.split('+')[0]
CUDA  = 'cu' + torch.version.cuda.replace('.', '')
!pip install torch-geometric -q
!pip install torch-scatter torch-sparse \
  -f https://data.pyg.org/whl/torch-{TORCH}+{CUDA}.html -q
!pip install scikit-learn matplotlib seaborn pandas numpy umap-learn -q
print('✅ All installed.')

In [ ]:
# ── CELL 2: Imports & Global Setup ────────────────────────
import os
os.environ['CUDA_LAUNCH_BLOCKING'] = '1'
import random, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
warnings.filterwarnings('ignore')

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import Adam
from torch_geometric.nn import GCNConv, GATConv
from sklearn.model_selection import KFold
from sklearn.metrics import (
    roc_auc_score, average_precision_score,
    roc_curve, precision_recall_curve)

SEED = 42
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device : {DEVICE}')
if DEVICE.type == 'cuda':
    print(f'GPU    : {torch.cuda.get_device_name(0)}')

plt.rcParams.update({
    'figure.dpi': 150,
    'font.size': 11,
    'font.family': 'DejaVu Sans',
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.grid': True,
    'grid.alpha': 0.25,
    'axes.linewidth': 0.8
})
C = ['#2E86AB','#E84855','#3BB273','#F7B731',
     '#9B59B6','#1ABC9C','#E67E22','#95A5A6']
print('✅ Imports done.')

In [ ]:
# ── CELL 3: Load HMDAD Dataset ────────────────────────────
try:
    from google.colab import files
    print('Upload HMDAD.txt ...')
    uploaded = files.upload()
    DATA_PATH = list(uploaded.keys())[0]
except Exception:
    DATA_PATH = 'HMDAD.txt'

df_raw  = pd.read_csv(DATA_PATH, sep='\t')
df_uniq = df_raw.drop_duplicates(subset=['Disease','Microbe'])
diseases = sorted(df_uniq['Disease'].unique())
microbes = sorted(df_uniq['Microbe'].unique())
d2i = {d:i for i,d in enumerate(diseases)}
m2i = {m:i for i,m in enumerate(microbes)}
nd, nm = len(diseases), len(microbes)

A = np.zeros((nd, nm), dtype=np.float32)
for _, row in df_uniq.iterrows():
    A[d2i[row['Disease']], m2i[row['Microbe']]] = 1.0

pos_d_all, pos_m_all = np.where(A == 1)
neg_d_all, neg_m_all = np.where(A == 0)

print(f'Diseases : {nd}')
print(f'Microbes : {nm}')
print(f'Positive pairs : {len(pos_d_all)}')
print(f'Negative pairs : {len(neg_d_all)}')
print(f'Density        : {A.mean()*100:.2f}%')

In [ ]:
# ── CELL 4: Similarity Computation ───────────────────────
def gip_similarity(profiles):
    norms = np.sum(profiles**2, axis=1)
    lam   = 1.0 / (np.mean(norms) + 1e-8)
    diff  = profiles[:,None,:] - profiles[None,:,:]
    return np.exp(-lam * np.sum(diff**2, axis=-1)).astype(np.float32)

def cosine_similarity(profiles):
    norms  = np.linalg.norm(profiles, axis=1, keepdims=True) + 1e-8
    normed = profiles / norms
    return np.clip((normed @ normed.T).astype(np.float32), 0, 1)

def integrated_similarity(profiles):
    return (gip_similarity(profiles) + cosine_similarity(profiles)) / 2.0

SD = integrated_similarity(A)
SM = integrated_similarity(A.T)
print(f'SD : {SD.shape}  SM : {SM.shape}')
print('✅ Similarities computed from full A (standard protocol).')

In [ ]:
# ── CELL 5: Graph Construction Utilities ─────────────────
def build_hetero_network(A_mat, SD_mat, SM_mat, threshold=0.3):
    n = nd + nm
    Y = np.zeros((n, n), dtype=np.float32)
    Y[:nd,:nd]  = SD_mat
    Y[:nd,nd:]  = A_mat
    Y[nd:,:nd]  = A_mat.T
    Y[nd:,nd:]  = SM_mat
    src, dst = [], []
    dr, mc = np.where(A_mat == 1)
    for d,m in zip(dr,mc):
        src += [int(d), int(nd+m)]
        dst += [int(nd+m), int(d)]
    r, c = np.where((SD_mat >= threshold) & (np.eye(nd)==0))
    for i,j in zip(r,c):
        src.append(int(i)); dst.append(int(j))
    for i in range(nm):
        s = SM_mat[i].copy(); s[i] = -1
        top = np.argsort(s)[-10:]
        for j in top:
            if SM_mat[i,j] >= 0.05:
                src.append(int(nd+i)); dst.append(int(nd+j))
    ei = torch.tensor([src,dst], dtype=torch.long)
    X  = torch.tensor(Y, dtype=torch.float32)
    return X, ei

def build_sim_networks(SD_mat, SM_mat, threshold=0.3):
    r, c = np.where((SD_mat >= threshold) & (np.eye(nd)==0))
    if len(r) == 0: r = c = np.arange(nd)
    ei_d = torch.tensor(np.stack([r,c]), dtype=torch.long)
    src_m, dst_m = [], []
    for i in range(nm):
        s = SM_mat[i].copy(); s[i] = -1
        top = np.argsort(s)[-10:]
        for j in top:
            if SM_mat[i,j] >= 0.05:
                src_m.append(i); dst_m.append(int(j))
    if len(src_m) == 0: src_m = dst_m = list(range(nm))
    ei_m = torch.tensor([src_m,dst_m], dtype=torch.long)
    return ei_d, ei_m

print('✅ Graph utilities defined.')

In [ ]:
# ── CELL 6: Model Definition ──────────────────────────────
#
# Architecture:
#   Parallel GCN + Sparse Adaptive GAT at every layer
#   Cross-level residual weighted aggregation (MADGAN)
#   Dual-view contrastive encoding (GCATCMDA)
#   Learnable sparsity threshold τ (NOVEL)

class SparseAdaptiveGAT(nn.Module):
    """
    GAT with learnable edge-gating threshold τ.
    gate = sigmoid(||h||_2 - |τ|)
    Output is gated by node-level feature norm vs learned threshold.
    τ starts at 0.1 and is updated via backpropagation.
    """
    def __init__(self, in_dim, out_dim, heads=4, dropout=0.4):
        super().__init__()
        assert out_dim % heads == 0
        self.gat  = GATConv(in_dim, out_dim//heads, heads=heads,
                            concat=True, dropout=dropout,
                            add_self_loops=True)
        self.bn   = nn.LayerNorm(out_dim)
        self.drop = dropout
        self.tau  = nn.Parameter(torch.tensor(0.1))  # learnable threshold

    def forward(self, x, edge_index):
        h    = self.gat(x, edge_index)
        gate = torch.sigmoid(h.norm(dim=-1, keepdim=True) - torch.abs(self.tau))
        h    = h * gate
        h    = F.elu(self.bn(h))
        return F.dropout(h, p=self.drop, training=self.training)

    def get_tau(self):
        return self.tau.item()


class CrossLevelEncoder(nn.Module):
    """
    At each layer:
      GCN(h)  and  SparseAdaptiveGAT(h) run in PARALLEL on same h
      h_fuse  = ELU( proj(h_gcn) + proj(h_gat) )  — additive fusion
      h_next  = h_gcn + h_gat + h_current          — residual
    Final z   = w[0]*h_fuse_0 + w[1]*h_fuse_1      — cross-level weights
    """
    def __init__(self, in_dim, hidden, out_dim,
                 n_layers=2, heads=4, dropout=0.4):
        super().__init__()
        self.n_layers = n_layers
        self.proj = nn.Sequential(
            nn.Linear(in_dim, hidden),
            nn.LayerNorm(hidden), nn.ReLU(), nn.Dropout(0.2))
        self.gcn_layers  = nn.ModuleList()
        self.gat_layers  = nn.ModuleList()
        self.bn_layers   = nn.ModuleList()
        self.proj_layers = nn.ModuleList()
        for i in range(n_layers):
            d_out = out_dim if i == n_layers-1 else hidden
            self.gcn_layers.append(GCNConv(hidden, d_out))
            self.gat_layers.append(
                SparseAdaptiveGAT(hidden, d_out, heads=heads, dropout=dropout))
            self.bn_layers.append(nn.LayerNorm(d_out))
            self.proj_layers.append(nn.Linear(d_out, out_dim))
        self.level_w = nn.Parameter(torch.ones(n_layers) / n_layers)
        self.drop    = dropout

    def forward(self, x, edge_index):
        h   = self.proj(x)
        w   = F.softmax(self.level_w, dim=0)
        out = []
        for i in range(self.n_layers):
            # Parallel branches — both see same h
            h_gcn = F.relu(self.bn_layers[i](self.gcn_layers[i](h, edge_index)))
            h_gcn = F.dropout(h_gcn, p=self.drop, training=self.training)
            h_gat = self.gat_layers[i](h, edge_index)
            # Additive fusion via shared projection
            p = self.proj_layers[i]
            h_fuse = F.elu(p(h_gcn) + p(h_gat))
            out.append(w[i] * h_fuse)
            # Residual carries both signals forward
            if h_gcn.shape[-1] == h.shape[-1]:
                h = h_gcn + h_gat + h
            else:
                h = h_gcn + h_gat
        return sum(out)

    def get_tau_values(self):
        """Return τ from each GAT layer."""
        return [layer.get_tau() for layer in self.gat_layers]


class SGAT_CLMDA(nn.Module):
    def __init__(self, nd, nm, feat_dim,
                 hidden=256, out_dim=64,
                 n_layers=2, heads=4, dropout=0.4,
                 use_contrastive=True):
        super().__init__()
        self.nd = nd; self.nm = nm
        self.use_contrastive = use_contrastive
        self.enc_A   = CrossLevelEncoder(feat_dim, hidden, out_dim, n_layers, heads, dropout)
        self.enc_B_d = CrossLevelEncoder(nd,       hidden, out_dim, n_layers, heads, dropout)
        self.enc_B_m = CrossLevelEncoder(nm,       hidden, out_dim, n_layers, heads, dropout)
        self.combine_d = nn.Sequential(
            nn.Linear(out_dim*2, out_dim), nn.LayerNorm(out_dim), nn.ReLU())
        self.combine_m = nn.Sequential(
            nn.Linear(out_dim*2, out_dim), nn.LayerNorm(out_dim), nn.ReLU())

    def encode(self, X, ei_h, FD, FM, ei_d, ei_m):
        z_A  = self.enc_A(X, ei_h)
        zd_A = z_A[:self.nd]; zm_A = z_A[self.nd:]
        zd_B = self.enc_B_d(FD, ei_d)
        zm_B = self.enc_B_m(FM, ei_m)
        zd = self.combine_d(torch.cat([zd_A, zd_B], dim=-1))
        zm = self.combine_m(torch.cat([zm_A, zm_B], dim=-1))
        return zd, zm, zd_A, zm_A, zd_B, zm_B

    def decode(self, zd, zm):
        return torch.sigmoid(zd @ zm.T)

    def forward(self, X, ei_h, FD, FM, ei_d, ei_m):
        zd, zm, zd_A, zm_A, zd_B, zm_B = self.encode(X, ei_h, FD, FM, ei_d, ei_m)
        return self.decode(zd, zm), zd, zm, zd_A, zm_A, zd_B, zm_B

    def get_tau_history(self):
        """Collect τ from all GAT layers in all encoders."""
        taus = {}
        for enc_name, enc in [('A', self.enc_A),
                                ('B_d', self.enc_B_d),
                                ('B_m', self.enc_B_m)]:
            for li, layer in enumerate(enc.gat_layers):
                taus[f'{enc_name}_L{li}'] = layer.get_tau()
        return taus


# GCN-only variant (ablation)
class GCNOnlyEncoder(nn.Module):
    def __init__(self, in_dim, hidden, out_dim, n_layers=2, dropout=0.4):
        super().__init__()
        self.proj = nn.Sequential(
            nn.Linear(in_dim, hidden),
            nn.LayerNorm(hidden), nn.ReLU(), nn.Dropout(0.2))
        self.gcn_layers = nn.ModuleList()
        self.bn_layers  = nn.ModuleList()
        self.level_w    = nn.Parameter(torch.ones(n_layers) / n_layers)
        self.drop = dropout
        for i in range(n_layers):
            d_out = out_dim if i == n_layers-1 else hidden
            self.gcn_layers.append(GCNConv(hidden, d_out))
            self.bn_layers.append(nn.LayerNorm(d_out))
    def forward(self, x, edge_index):
        h = self.proj(x)
        w = F.softmax(self.level_w, dim=0)
        out = []
        for i in range(len(self.gcn_layers)):
            h = F.relu(self.bn_layers[i](self.gcn_layers[i](h, edge_index)))
            h = F.dropout(h, p=self.drop, training=self.training)
            out.append(w[i] * h)
        return sum(out)


class SGAT_CLMDA_GCNOnly(nn.Module):
    """Ablation: GCN only, no GAT, no contrastive."""
    def __init__(self, nd, nm, feat_dim,
                 hidden=256, out_dim=64, n_layers=2, dropout=0.4):
        super().__init__()
        self.nd = nd; self.nm = nm
        self.enc_A   = GCNOnlyEncoder(feat_dim, hidden, out_dim, n_layers, dropout)
        self.enc_B_d = GCNOnlyEncoder(nd,       hidden, out_dim, n_layers, dropout)
        self.enc_B_m = GCNOnlyEncoder(nm,       hidden, out_dim, n_layers, dropout)
        self.combine_d = nn.Sequential(
            nn.Linear(out_dim*2, out_dim), nn.LayerNorm(out_dim), nn.ReLU())
        self.combine_m = nn.Sequential(
            nn.Linear(out_dim*2, out_dim), nn.LayerNorm(out_dim), nn.ReLU())
    def forward(self, X, ei_h, FD, FM, ei_d, ei_m):
        z_A  = self.enc_A(X, ei_h)
        zd_A = z_A[:self.nd]; zm_A = z_A[self.nd:]
        zd_B = self.enc_B_d(FD, ei_d)
        zm_B = self.enc_B_m(FM, ei_m)
        zd = self.combine_d(torch.cat([zd_A, zd_B], dim=-1))
        zm = self.combine_m(torch.cat([zm_A, zm_B], dim=-1))
        return torch.sigmoid(zd @ zm.T), zd, zm, zd_A, zm_A, zd_B, zm_B


print('✅ Model classes defined.')
print('   SGAT_CLMDA        : Full model (parallel GCN+GAT, contrastive)')
print('   SGAT_CLMDA        : use_contrastive=False → ablation without CL')
print('   SGAT_CLMDA_GCNOnly: ablation GCN-only')

In [ ]:
# ── CELL 7: Loss Functions ────────────────────────────────
def bce_loss(pred, target):
    n_pos = target.sum().clamp(min=1)
    n_neg = (1-target).sum().clamp(min=1)
    pw    = (n_neg/n_pos).clamp(max=15.0)
    w     = torch.where(target==1,
                        pw*torch.ones_like(target),
                        torch.ones_like(target))
    return F.binary_cross_entropy(pred, target, weight=w)

def nt_xent_loss(z1, z2, tau=0.5):
    z1 = F.normalize(z1, dim=-1)
    z2 = F.normalize(z2, dim=-1)
    n  = z1.shape[0]
    logits = (z1 @ z2.T) / tau
    labels = torch.arange(n, device=z1.device)
    return (F.cross_entropy(logits, labels) +
            F.cross_entropy(logits.T, labels)) / 2.0

def total_loss(pred, target, zd_A, zd_B, zm_A, zm_B,
               lam=0.1, tau=0.5, use_contrastive=True):
    l_bce = bce_loss(pred, target)
    if not use_contrastive:
        return l_bce
    l_cl = (nt_xent_loss(zd_A, zd_B, tau) +
            nt_xent_loss(zm_A, zm_B, tau)) / 2
    return l_bce + lam * l_cl

print('✅ Loss functions defined.')

In [ ]:
# ── CELL 8: Main 5-Fold Cross Validation ─────────────────
# Tracks: AUC, AUPR, training loss, tau evolution per fold

EPOCHS = 400; LR = 1e-3; HIDDEN = 256; OUT_DIM = 64
N_LAYERS = 2; HEADS = 4; DROPOUT = 0.5
THRESHOLD = 0.3; LAM_CL = 0.1; TAU = 0.5
K_FOLDS = 5; PATIENCE = 50; FEAT_DIM = nd + nm

all_auc, all_aupr   = [], []
all_fpr,  all_tpr   = [], []
all_prec, all_rec   = [], []
fold_train_losses   = []
fold_tau_histories  = []   # tau tracked per epoch per fold
best_model_state    = None
best_auc            = 0.0

kf  = KFold(n_splits=K_FOLDS, shuffle=True, random_state=SEED)
pos_indices = np.arange(len(pos_d_all))

for fold, (tr_idx, te_idx) in enumerate(kf.split(pos_indices)):
    print(f'\n── Fold {fold+1}/{K_FOLDS} ──────────────')

    tr_pd = pos_d_all[tr_idx]; tr_pm = pos_m_all[tr_idx]
    te_pd = pos_d_all[te_idx]; te_pm = pos_m_all[te_idx]

    A_tr = np.zeros((nd, nm), dtype=np.float32)
    A_tr[tr_pd, tr_pm] = 1.0

    X_tr, ei_h = build_hetero_network(A_tr, SD, SM, THRESHOLD)
    ei_d, ei_m = build_sim_networks(SD, SM, THRESHOLD)
    FD = torch.tensor(SD, dtype=torch.float32)
    FM = torch.tensor(SM, dtype=torch.float32)

    X_tr = X_tr.to(DEVICE); ei_h = ei_h.to(DEVICE)
    FD   = FD.to(DEVICE);   FM   = FM.to(DEVICE)
    ei_d = ei_d.to(DEVICE); ei_m = ei_m.to(DEVICE)

    np.random.seed(SEED + fold)
    neg_sel = np.random.choice(len(neg_d_all), size=len(tr_pd), replace=False)
    all_d = torch.tensor(np.concatenate([tr_pd, neg_d_all[neg_sel]]),
                         dtype=torch.long).to(DEVICE)
    all_m = torch.tensor(np.concatenate([tr_pm, neg_m_all[neg_sel]]),
                         dtype=torch.long).to(DEVICE)
    all_y = torch.tensor(np.concatenate([np.ones(len(tr_pd)),
                                          np.zeros(len(tr_pd))]),
                         dtype=torch.float32).to(DEVICE)

    model = SGAT_CLMDA(nd, nm, FEAT_DIM, HIDDEN, OUT_DIM,
                       N_LAYERS, HEADS, DROPOUT).to(DEVICE)
    opt   = Adam(model.parameters(), lr=LR, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
                opt, T_0=80, T_mult=2, eta_min=1e-5)

    losses     = []
    tau_hist   = []   # mean tau across all GAT layers per epoch
    best_loss  = float('inf')
    pat_cnt    = 0

    model.train()
    for epoch in range(EPOCHS):
        opt.zero_grad()
        scores, zd, zm, zd_A, zm_A, zd_B, zm_B = model(
            X_tr, ei_h, FD, FM, ei_d, ei_m)
        pred = scores[all_d, all_m]
        loss = total_loss(pred, all_y, zd_A, zd_B, zm_A, zm_B,
                          LAM_CL, TAU, use_contrastive=True)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step()
        sched.step(epoch+1)
        losses.append(loss.item())

        # Track mean tau across all GAT layers
        tau_vals = list(model.get_tau_history().values())
        tau_hist.append(np.mean(tau_vals))

        if loss.item() < best_loss - 1e-4:
            best_loss = loss.item(); pat_cnt = 0
        else:
            pat_cnt += 1
        if pat_cnt >= PATIENCE:
            print(f'  Early stop @ epoch {epoch+1}')
            break
        if (epoch+1) % 100 == 0:
            print(f'  Epoch {epoch+1:3d} | loss={loss.item():.4f} '
                  f'τ_mean={np.mean(tau_vals):.4f} '
                  f'lr={opt.param_groups[0]["lr"]:.5f}')

    fold_train_losses.append(losses)
    fold_tau_histories.append(tau_hist)

    # Evaluation
    model.eval()
    with torch.no_grad():
        sc_full, *_ = model(X_tr, ei_h, FD, FM, ei_d, ei_m)
    sc_np = sc_full.cpu().numpy()

    te_scores = sc_np[te_pd, te_pm]
    A_te = np.zeros((nd, nm), dtype=np.float32)
    A_te[te_pd, te_pm] = 1.0
    neg_td, neg_tm = np.where((A==0) & (A_te==0))
    np.random.seed(SEED + fold + 100)
    nidx = np.random.choice(len(neg_td), size=len(te_pd), replace=False)
    neg_scores = sc_np[neg_td[nidx], neg_tm[nidx]]

    y_true  = np.concatenate([np.ones(len(te_scores)),  np.zeros(len(neg_scores))])
    y_score = np.concatenate([te_scores, neg_scores])

    auc  = roc_auc_score(y_true, y_score)
    aupr = average_precision_score(y_true, y_score)
    fpr, tpr, _ = roc_curve(y_true, y_score)
    prec, rec, _ = precision_recall_curve(y_true, y_score)

    all_auc.append(auc); all_aupr.append(aupr)
    all_fpr.append(fpr); all_tpr.append(tpr)
    all_prec.append(prec); all_rec.append(rec)
    print(f'  ✅ Fold {fold+1} → AUC={auc:.4f}  AUPR={aupr:.4f}')

    if auc > best_auc:
        best_auc = auc
        best_model_state = {k: v.cpu().clone()
                            for k, v in model.state_dict().items()}

    del model, opt, X_tr, ei_h, FD, FM, ei_d, ei_m
    torch.cuda.empty_cache()

print(f'\n{"="*50}')
print('SGAT-CLMDA  5-Fold CV Results:')
print(f'  AUC  = {np.mean(all_auc):.4f} ± {np.std(all_auc):.4f}')
print(f'  AUPR = {np.mean(all_aupr):.4f} ± {np.std(all_aupr):.4f}')
print('='*50)

In [ ]:
# ── CELL 9: Leakage Protocol Comparison ──────────────────
# Run the SAME model under two protocols:
#   Honest  : graph edges from training A only (what we do)
#   Leaky   : graph edges from full A (what MADGAN/DBGCNMDA do)
# Shows the inflation caused by leakage.

print('Running leakage comparison experiment...')
print('(2-fold only for speed — demonstrates the gap clearly)\n')

LEAKAGE_FOLDS = 2
kf2 = KFold(n_splits=5, shuffle=True, random_state=SEED)

leak_results  = {'honest': [], 'leaky': []}

for fold, (tr_idx, te_idx) in enumerate(
        list(kf2.split(pos_indices))[:LEAKAGE_FOLDS]):

    tr_pd = pos_d_all[tr_idx]; tr_pm = pos_m_all[tr_idx]
    te_pd = pos_d_all[te_idx]; te_pm = pos_m_all[te_idx]

    A_tr = np.zeros((nd, nm), dtype=np.float32)
    A_tr[tr_pd, tr_pm] = 1.0

    for protocol in ['honest', 'leaky']:
        # Honest: build graph from A_tr only
        # Leaky : build graph from full A (includes test pairs)
        A_for_graph = A_tr if protocol == 'honest' else A

        X_p, ei_h_p = build_hetero_network(A_for_graph, SD, SM, THRESHOLD)
        ei_d_p, ei_m_p = build_sim_networks(SD, SM, THRESHOLD)
        FD_p = torch.tensor(SD, dtype=torch.float32).to(DEVICE)
        FM_p = torch.tensor(SM, dtype=torch.float32).to(DEVICE)
        X_p  = X_p.to(DEVICE); ei_h_p = ei_h_p.to(DEVICE)
        ei_d_p = ei_d_p.to(DEVICE); ei_m_p = ei_m_p.to(DEVICE)

        np.random.seed(SEED + fold)
        neg_sel = np.random.choice(len(neg_d_all), size=len(tr_pd), replace=False)
        d_t = torch.tensor(np.concatenate([tr_pd, neg_d_all[neg_sel]]),
                           dtype=torch.long).to(DEVICE)
        m_t = torch.tensor(np.concatenate([tr_pm, neg_m_all[neg_sel]]),
                           dtype=torch.long).to(DEVICE)
        y_t = torch.tensor(np.concatenate([np.ones(len(tr_pd)),
                                            np.zeros(len(tr_pd))]),
                           dtype=torch.float32).to(DEVICE)

        m = SGAT_CLMDA(nd, nm, FEAT_DIM, HIDDEN, OUT_DIM,
                       N_LAYERS, HEADS, DROPOUT).to(DEVICE)
        o = Adam(m.parameters(), lr=LR, weight_decay=1e-4)
        s = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(o, T_0=80, T_mult=2)

        best_l = float('inf'); pat = 0
        m.train()
        for ep in range(EPOCHS):
            o.zero_grad()
            sc, zd, zm, zd_A, zm_A, zd_B, zm_B = m(
                X_p, ei_h_p, FD_p, FM_p, ei_d_p, ei_m_p)
            pred = sc[d_t, m_t]
            loss = total_loss(pred, y_t, zd_A, zd_B, zm_A, zm_B,
                              LAM_CL, TAU)
            loss.backward()
            nn.utils.clip_grad_norm_(m.parameters(), 1.0)
            o.step(); s.step(ep+1)
            if loss.item() < best_l - 1e-4: best_l = loss.item(); pat = 0
            else: pat += 1
            if pat >= PATIENCE: break

        m.eval()
        with torch.no_grad():
            sc_full, *_ = m(X_p, ei_h_p, FD_p, FM_p, ei_d_p, ei_m_p)
        sc_np = sc_full.cpu().numpy()

        te_scores = sc_np[te_pd, te_pm]
        A_te2 = np.zeros((nd, nm), dtype=np.float32)
        A_te2[te_pd, te_pm] = 1.0
        ntd, ntm = np.where((A==0) & (A_te2==0))
        np.random.seed(SEED + fold + 200)
        ni = np.random.choice(len(ntd), size=len(te_pd), replace=False)
        ng = sc_np[ntd[ni], ntm[ni]]

        yt = np.concatenate([np.ones(len(te_scores)), np.zeros(len(ng))])
        ys = np.concatenate([te_scores, ng])
        auc_val = roc_auc_score(yt, ys)
        leak_results[protocol].append(auc_val)
        print(f'  Fold {fold+1} [{protocol:6s}] AUC = {auc_val:.4f}')

        del m, o, X_p, ei_h_p, FD_p, FM_p, ei_d_p, ei_m_p
        torch.cuda.empty_cache()

honest_mean = np.mean(leak_results['honest'])
leaky_mean  = np.mean(leak_results['leaky'])
print(f'\nHonest protocol AUC : {honest_mean:.4f}')
print(f'Leaky  protocol AUC : {leaky_mean:.4f}')
print(f'Leakage gap         : +{leaky_mean - honest_mean:.4f}')

In [ ]:
# ── CELL 10: Ablation Study ───────────────────────────────
# Compares four configurations:
#   (A) Full SGAT-CLMDA  : parallel GCN+GAT + contrastive learning
#   (B) No Contrastive   : parallel GCN+GAT, BCE only
#   (C) GCN Only         : no GAT, no contrastive
#   (D) GAT Only         : no GCN (uses GAT encoder directly)
# All on same 2 folds for fair comparison.

print('Running ablation study...')
print('Configurations: Full | No-CL | GCN-Only | GAT-Only\n')

ABL_FOLDS = 2
kf3 = KFold(n_splits=5, shuffle=True, random_state=SEED)

ablation_results = {
    'Full (GCN+GAT+CL)': [],
    'No Contrastive':    [],
    'GCN Only':          [],
    'GAT Only':          [],
}

# GAT-only encoder — same interface as CrossLevelEncoder
class GATOnlyEncoder(nn.Module):
    def __init__(self, in_dim, hidden, out_dim, n_layers=2, heads=4, dropout=0.4):
        super().__init__()
        self.proj = nn.Sequential(
            nn.Linear(in_dim, hidden),
            nn.LayerNorm(hidden), nn.ReLU(), nn.Dropout(0.2))
        self.gat_layers = nn.ModuleList()
        self.level_w    = nn.Parameter(torch.ones(n_layers)/n_layers)
        self.drop = dropout
        for i in range(n_layers):
            d_out = out_dim if i==n_layers-1 else hidden
            assert d_out % heads == 0
            self.gat_layers.append(
                GATConv(hidden, d_out//heads, heads=heads,
                        concat=True, dropout=dropout, add_self_loops=True))
    def forward(self, x, edge_index):
        h = self.proj(x)
        w = F.softmax(self.level_w, dim=0)
        out = []
        for i, layer in enumerate(self.gat_layers):
            h = F.elu(layer(h, edge_index))
            h = F.dropout(h, p=self.drop, training=self.training)
            out.append(w[i] * h)
        return sum(out)

class SGAT_CLMDA_GATOnly(nn.Module):
    def __init__(self, nd, nm, feat_dim,
                 hidden=256, out_dim=64, n_layers=2, heads=4, dropout=0.4):
        super().__init__()
        self.nd = nd; self.nm = nm
        self.enc_A   = GATOnlyEncoder(feat_dim, hidden, out_dim, n_layers, heads, dropout)
        self.enc_B_d = GATOnlyEncoder(nd,       hidden, out_dim, n_layers, heads, dropout)
        self.enc_B_m = GATOnlyEncoder(nm,       hidden, out_dim, n_layers, heads, dropout)
        self.combine_d = nn.Sequential(
            nn.Linear(out_dim*2, out_dim), nn.LayerNorm(out_dim), nn.ReLU())
        self.combine_m = nn.Sequential(
            nn.Linear(out_dim*2, out_dim), nn.LayerNorm(out_dim), nn.ReLU())
    def forward(self, X, ei_h, FD, FM, ei_d, ei_m):
        z_A  = self.enc_A(X, ei_h)
        zd_A = z_A[:self.nd]; zm_A = z_A[self.nd:]
        zd_B = self.enc_B_d(FD, ei_d)
        zm_B = self.enc_B_m(FM, ei_m)
        zd = self.combine_d(torch.cat([zd_A, zd_B], dim=-1))
        zm = self.combine_m(torch.cat([zm_A, zm_B], dim=-1))
        return torch.sigmoid(zd @ zm.T), zd, zm, zd_A, zm_A, zd_B, zm_B


def run_variant(model_fn, use_cl, fold_data, fold_idx):
    tr_pd, tr_pm, te_pd, te_pm, X_p, ei_h, FD_p, FM_p, ei_d, ei_m = fold_data
    np.random.seed(SEED + fold_idx)
    neg_sel = np.random.choice(len(neg_d_all), size=len(tr_pd), replace=False)
    d_t = torch.tensor(np.concatenate([tr_pd, neg_d_all[neg_sel]]),
                       dtype=torch.long).to(DEVICE)
    m_t = torch.tensor(np.concatenate([tr_pm, neg_m_all[neg_sel]]),
                       dtype=torch.long).to(DEVICE)
    y_t = torch.tensor(np.concatenate([np.ones(len(tr_pd)),
                                        np.zeros(len(tr_pd))]),
                       dtype=torch.float32).to(DEVICE)
    m = model_fn().to(DEVICE)
    o = Adam(m.parameters(), lr=LR, weight_decay=1e-4)
    s = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(o, T_0=80, T_mult=2)
    best_l = float('inf'); pat = 0
    m.train()
    for ep in range(EPOCHS):
        o.zero_grad()
        sc, zd, zm, zd_A, zm_A, zd_B, zm_B = m(
            X_p, ei_h, FD_p, FM_p, ei_d, ei_m)
        pred = sc[d_t, m_t]
        loss = total_loss(pred, y_t, zd_A, zd_B, zm_A, zm_B,
                          LAM_CL, TAU, use_contrastive=use_cl)
        loss.backward()
        nn.utils.clip_grad_norm_(m.parameters(), 1.0)
        o.step(); s.step(ep+1)
        if loss.item() < best_l - 1e-4: best_l = loss.item(); pat = 0
        else: pat += 1
        if pat >= PATIENCE: break
    m.eval()
    with torch.no_grad():
        sc_full, *_ = m(X_p, ei_h, FD_p, FM_p, ei_d, ei_m)
    sc_np = sc_full.cpu().numpy()
    te_scores = sc_np[te_pd, te_pm]
    A_te3 = np.zeros((nd, nm), dtype=np.float32)
    A_te3[te_pd, te_pm] = 1.0
    ntd, ntm = np.where((A==0) & (A_te3==0))
    np.random.seed(SEED + fold_idx + 300)
    ni = np.random.choice(len(ntd), size=len(te_pd), replace=False)
    ng = sc_np[ntd[ni], ntm[ni]]
    yt = np.concatenate([np.ones(len(te_scores)), np.zeros(len(ng))])
    ys = np.concatenate([te_scores, ng])
    auc_val = roc_auc_score(yt, ys)
    del m, o
    torch.cuda.empty_cache()
    return auc_val

for fold_idx, (tr_idx, te_idx) in enumerate(
        list(kf3.split(pos_indices))[:ABL_FOLDS]):

    tr_pd = pos_d_all[tr_idx]; tr_pm = pos_m_all[tr_idx]
    te_pd = pos_d_all[te_idx]; te_pm = pos_m_all[te_idx]
    A_tr = np.zeros((nd, nm), dtype=np.float32)
    A_tr[tr_pd, tr_pm] = 1.0
    X_p, ei_h = build_hetero_network(A_tr, SD, SM, THRESHOLD)
    ei_d, ei_m = build_sim_networks(SD, SM, THRESHOLD)
    FD_p = torch.tensor(SD, dtype=torch.float32).to(DEVICE)
    FM_p = torch.tensor(SM, dtype=torch.float32).to(DEVICE)
    X_p = X_p.to(DEVICE); ei_h = ei_h.to(DEVICE)
    ei_d = ei_d.to(DEVICE); ei_m = ei_m.to(DEVICE)
    fd = (tr_pd, tr_pm, te_pd, te_pm, X_p, ei_h, FD_p, FM_p, ei_d, ei_m)

    print(f'Fold {fold_idx+1}:')

    # A: Full model
    fn = lambda: SGAT_CLMDA(nd, nm, FEAT_DIM, HIDDEN, OUT_DIM, N_LAYERS, HEADS, DROPOUT)
    auc = run_variant(fn, True, fd, fold_idx)
    ablation_results['Full (GCN+GAT+CL)'].append(auc)
    print(f'  Full (GCN+GAT+CL) : AUC={auc:.4f}')

    # B: No contrastive
    fn = lambda: SGAT_CLMDA(nd, nm, FEAT_DIM, HIDDEN, OUT_DIM, N_LAYERS, HEADS, DROPOUT)
    auc = run_variant(fn, False, fd, fold_idx)
    ablation_results['No Contrastive'].append(auc)
    print(f'  No Contrastive    : AUC={auc:.4f}')

    # C: GCN only
    fn = lambda: SGAT_CLMDA_GCNOnly(nd, nm, FEAT_DIM, HIDDEN, OUT_DIM, N_LAYERS)
    auc = run_variant(fn, False, fd, fold_idx)
    ablation_results['GCN Only'].append(auc)
    print(f'  GCN Only          : AUC={auc:.4f}')

    # D: GAT only
    fn = lambda: SGAT_CLMDA_GATOnly(nd, nm, FEAT_DIM, HIDDEN, OUT_DIM, N_LAYERS, HEADS)
    auc = run_variant(fn, False, fd, fold_idx)
    ablation_results['GAT Only'].append(auc)
    print(f'  GAT Only          : AUC={auc:.4f}')

    del X_p, ei_h, FD_p, FM_p, ei_d, ei_m
    torch.cuda.empty_cache()

print('\nAblation Summary (mean over 2 folds):')
for k, v in ablation_results.items():
    print(f'  {k:<24} AUC = {np.mean(v):.4f}')

In [ ]:
# ── CELL 11: Plot — Adaptive Threshold τ Evolution ───────
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Left: τ per fold over epochs
ax = axes[0]
for i, tau_h in enumerate(fold_tau_histories):
    ax.plot(tau_h, lw=1.5, alpha=0.85, color=C[i], label=f'Fold {i+1}')
ax.set_xlabel('Training Epoch')
ax.set_ylabel('Mean τ (adaptive threshold)')
ax.set_title('Adaptive Threshold τ During Training', fontweight='bold')
ax.legend(fontsize=9)
ax.axhline(0.1, color='grey', linestyle=':', lw=1.2, label='Init τ=0.1')

# Right: τ final values across folds
ax2 = axes[1]
final_taus = [th[-1] for th in fold_tau_histories]
bars = ax2.bar([f'Fold {i+1}' for i in range(K_FOLDS)],
               final_taus, color=C[:K_FOLDS], edgecolor='white', width=0.5)
ax2.axhline(0.1, color='grey', linestyle=':', lw=1.5, label='Init τ=0.1')
for bar, v in zip(bars, final_taus):
    ax2.text(bar.get_x() + bar.get_width()/2, v + 0.002,
             f'{v:.4f}', ha='center', va='bottom', fontsize=9)
ax2.set_ylabel('Final τ value')
ax2.set_title('Final Learned τ Per Fold', fontweight='bold')
ax2.legend(fontsize=9)

plt.suptitle('Sparse Adaptive GAT: Learned Threshold τ Behaviour',
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('plot_tau_evolution.png', dpi=200, bbox_inches='tight')
plt.show()
print(f'τ init=0.1 | final mean={np.mean(final_taus):.4f} ± {np.std(final_taus):.4f}')

In [ ]:
# ── CELL 12: Plot — Leakage Protocol Comparison ──────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Left: per-fold bar comparison
ax = axes[0]
x = np.arange(LEAKAGE_FOLDS)
w = 0.3
ax.bar(x - w/2, leak_results['honest'], w,
       label='Honest Protocol', color=C[0], edgecolor='white')
ax.bar(x + w/2, leak_results['leaky'],  w,
       label='Leaky Protocol',  color=C[1], edgecolor='white', alpha=0.8)
for xi, (h, l) in enumerate(zip(leak_results['honest'], leak_results['leaky'])):
    ax.text(xi - w/2, h + 0.003, f'{h:.4f}', ha='center', fontsize=8)
    ax.text(xi + w/2, l + 0.003, f'{l:.4f}', ha='center', fontsize=8)
ax.set_xticks(x); ax.set_xticklabels([f'Fold {i+1}' for i in range(LEAKAGE_FOLDS)])
ax.set_ylabel('AUC'); ax.set_ylim(0.82, 1.02)
ax.set_title('Per-Fold AUC: Honest vs Leaky Protocol', fontweight='bold')
ax.legend(fontsize=10)

# Right: mean comparison with gap annotation
ax2 = axes[1]
labels = ['Honest\n(Our Protocol)', 'Leaky\n(Baseline Papers)']
vals   = [honest_mean, leaky_mean]
bars   = ax2.bar(labels, vals, color=[C[0], C[1]],
                 edgecolor='white', width=0.45)
for bar, v in zip(bars, vals):
    ax2.text(bar.get_x() + bar.get_width()/2, v + 0.003,
             f'{v:.4f}', ha='center', fontsize=11, fontweight='bold')
ax2.annotate('',
    xy=(1, leaky_mean), xytext=(0, honest_mean),
    arrowprops=dict(arrowstyle='<->', color='black', lw=1.5))
ax2.text(0.5, (honest_mean + leaky_mean)/2 + 0.005,
         f'Gap = +{leaky_mean - honest_mean:.4f}',
         ha='center', fontsize=10, color='black', fontweight='bold')
ax2.set_ylabel('Mean AUC'); ax2.set_ylim(0.82, 1.02)
ax2.set_title('Mean AUC: Protocol Comparison', fontweight='bold')

plt.suptitle('Data Leakage Effect on Reported AUC\n'
             '(Same model, same folds — only evaluation protocol differs)',
             fontsize=12, fontweight='bold', y=1.03)
plt.tight_layout()
plt.savefig('plot_leakage_comparison.png', dpi=200, bbox_inches='tight')
plt.show()

In [ ]:
# ── CELL 13: Plot — Ablation Study ───────────────────────
configs = list(ablation_results.keys())
means   = [np.mean(v) for v in ablation_results.values()]
full_auc = means[0]
abl_colors = [C[2] if i==0 else C[7] for i in range(len(configs))]

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Left: horizontal bar chart
ax = axes[0]
bars = ax.barh(configs[::-1], means[::-1],
               color=abl_colors[::-1], edgecolor='white', height=0.5)
for bar, v in zip(bars, means[::-1]):
    ax.text(v + 0.001, bar.get_y() + bar.get_height()/2,
            f'{v:.4f}', va='center', fontsize=10)
ax.axvline(full_auc, color=C[2], linestyle='--', lw=1.5,
           label=f'Full model: {full_auc:.4f}')
ax.set_xlabel('AUC'); ax.set_xlim(0.82, 1.0)
ax.set_title('Ablation: AUC Comparison', fontweight='bold')
ax.legend(fontsize=9)

# Right: delta from full model
ax2 = axes[1]
deltas = [v - full_auc for v in means]
delta_colors = [C[1] if d < 0 else C[2] for d in deltas]
bars2 = ax2.bar(range(len(configs)), deltas,
                color=delta_colors, edgecolor='white', width=0.5)
ax2.axhline(0, color='black', lw=0.8)
ax2.set_xticks(range(len(configs)))
ax2.set_xticklabels(['Full\n(GCN+GAT\n+CL)',
                     'No\nContrastive',
                     'GCN\nOnly',
                     'GAT\nOnly'], fontsize=9)
for bar, d in zip(bars2, deltas):
    ypos = d + 0.001 if d >= 0 else d - 0.003
    ax2.text(bar.get_x() + bar.get_width()/2, ypos,
             f'{d:+.4f}', ha='center', fontsize=9, fontweight='bold')
ax2.set_ylabel('ΔAUC vs Full Model')
ax2.set_title('Contribution of Each Component\n(Negative = removing it hurts)',
              fontweight='bold')

plt.suptitle('Ablation Study: Component Contribution to AUC',
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('plot_ablation.png', dpi=200, bbox_inches='tight')
plt.show()

print('\nComponent contribution summary:')
for k, d in zip(configs[1:], deltas[1:]):
    comp = k.replace('Full (GCN+GAT+CL) - ','')
    print(f'  Removing {k:<22}: ΔAUC = {d:+.4f}')

In [ ]:
# ── CELL 14: Plot — ROC Curves ───────────────────────────
mean_fpr = np.linspace(0, 1, 200)
tprs_i   = [np.interp(mean_fpr, f, t) for f, t in zip(all_fpr, all_tpr)]
mean_tpr = np.mean(tprs_i, axis=0)
std_tpr  = np.std(tprs_i,  axis=0)

fig, ax = plt.subplots(figsize=(6.5, 6))
for i, (fpr, tpr) in enumerate(zip(all_fpr, all_tpr)):
    ax.plot(fpr, tpr, lw=1.0, alpha=0.35, color=C[i],
            label=f'Fold {i+1} (AUC={all_auc[i]:.4f})')
ax.plot(mean_fpr, mean_tpr, lw=2.5, color='black',
        label=f'Mean (AUC={np.mean(all_auc):.4f}±{np.std(all_auc):.4f})')
ax.fill_between(mean_fpr, mean_tpr-std_tpr, mean_tpr+std_tpr,
                alpha=0.10, color='black', label='±1 Std Dev')
ax.plot([0,1],[0,1],'--', color='grey', lw=1, label='Random')
ax.set_xlabel('False Positive Rate'); ax.set_ylabel('True Positive Rate')
ax.set_title('SGAT-CLMDA — ROC Curves (5-Fold CV)', fontweight='bold')
ax.legend(fontsize=8.5, loc='lower right')
plt.tight_layout()
plt.savefig('plot_roc.png', dpi=200, bbox_inches='tight')
plt.show()

In [ ]:
# ── CELL 15: Plot — PR Curves ────────────────────────────
fig, ax = plt.subplots(figsize=(6.5, 6))
for i, (prec, rec) in enumerate(zip(all_prec, all_rec)):
    ax.plot(rec, prec, lw=1.2, alpha=0.5, color=C[i],
            label=f'Fold {i+1} (AUPR={all_aupr[i]:.4f})')
ax.set_xlabel('Recall'); ax.set_ylabel('Precision')
ax.set_title(f'SGAT-CLMDA — PR Curves\n'
             f'Mean AUPR={np.mean(all_aupr):.4f}±{np.std(all_aupr):.4f}',
             fontweight='bold')
ax.legend(fontsize=8.5); ax.set_xlim(0,1); ax.set_ylim(0,1)
plt.tight_layout()
plt.savefig('plot_pr.png', dpi=200, bbox_inches='tight')
plt.show()

In [ ]:
# ── CELL 16: Plot — Training Loss ────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

ax = axes[0]
for i, losses in enumerate(fold_train_losses):
    ax.plot(losses, lw=1.3, alpha=0.75, color=C[i], label=f'Fold {i+1}')
ax.set_xlabel('Epoch'); ax.set_ylabel('Loss')
ax.set_title('Training Loss per Fold', fontweight='bold')
ax.legend(fontsize=9)

ax2 = axes[1]
max_ep = max(len(l) for l in fold_train_losses)
padded = [l + [l[-1]]*(max_ep-len(l)) for l in fold_train_losses]
mean_l = np.mean(padded, axis=0)
std_l  = np.std(padded,  axis=0)
ep_arr = np.arange(max_ep)
ax2.plot(ep_arr, mean_l, lw=2.2, color=C[0], label='Mean loss')
ax2.fill_between(ep_arr, mean_l-std_l, mean_l+std_l,
                 alpha=0.15, color=C[0], label='±1 Std')
ax2.set_xlabel('Epoch'); ax2.set_ylabel('Loss')
ax2.set_title('Mean Training Loss ± Std Dev', fontweight='bold')
ax2.legend(fontsize=9)

plt.suptitle('Training Convergence — SGAT-CLMDA',
             fontsize=12, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('plot_loss.png', dpi=200, bbox_inches='tight')
plt.show()

In [ ]:
# ── CELL 17: Train Final Model on All Data ───────────────
print('Training final model on all 450 associations...')
X_full, ei_full = build_hetero_network(A, SD, SM, THRESHOLD)
ei_d_f, ei_m_f  = build_sim_networks(SD, SM, THRESHOLD)
FD_f = torch.tensor(SD, dtype=torch.float32).to(DEVICE)
FM_f = torch.tensor(SM, dtype=torch.float32).to(DEVICE)
X_full  = X_full.to(DEVICE);  ei_full  = ei_full.to(DEVICE)
ei_d_f  = ei_d_f.to(DEVICE);  ei_m_f   = ei_m_f.to(DEVICE)

final_model = SGAT_CLMDA(nd, nm, FEAT_DIM, HIDDEN, OUT_DIM,
                          N_LAYERS, HEADS, DROPOUT).to(DEVICE)
if best_model_state:
    final_model.load_state_dict(best_model_state, strict=False)

opt_f = Adam(final_model.parameters(), lr=LR, weight_decay=1e-4)
np.random.seed(SEED)
neg_f = np.random.choice(len(neg_d_all), size=len(pos_d_all), replace=False)
fd_all = torch.tensor(np.concatenate([pos_d_all, neg_d_all[neg_f]]),
                      dtype=torch.long).to(DEVICE)
fm_all = torch.tensor(np.concatenate([pos_m_all, neg_m_all[neg_f]]),
                      dtype=torch.long).to(DEVICE)
fy_all = torch.tensor(np.concatenate([np.ones(len(pos_d_all)),
                                       np.zeros(len(pos_d_all))]),
                      dtype=torch.float32).to(DEVICE)

final_model.train()
for ep in range(200):
    opt_f.zero_grad()
    sc, zd, zm, zd_A, zm_A, zd_B, zm_B = final_model(
        X_full, ei_full, FD_f, FM_f, ei_d_f, ei_m_f)
    pred = sc[fd_all, fm_all]
    loss = total_loss(pred, fy_all, zd_A, zd_B, zm_A, zm_B, LAM_CL, TAU)
    loss.backward()
    nn.utils.clip_grad_norm_(final_model.parameters(), 1.0)
    opt_f.step()
    if (ep+1) % 50 == 0:
        print(f'  Epoch {ep+1}: loss={loss.item():.4f}')

final_model.eval()
with torch.no_grad():
    final_scores, *_ = final_model(
        X_full, ei_full, FD_f, FM_f, ei_d_f, ei_m_f)
final_np = final_scores.cpu().numpy()
print(f'Score matrix: {final_np.shape} ✅')

In [ ]:
# ── CELL 18: Case Study — Top 10 per Disease ─────────────
def get_top_k(disease_name, k=10):
    if disease_name not in d2i: return None
    di  = d2i[disease_name]
    row = final_np[di].copy()
    row[A[di] == 1] = -1.0   # exclude known
    top = np.argsort(row)[::-1][:k]
    return pd.DataFrame({
        'Rank':    range(1, k+1),
        'Microbe': [microbes[i] for i in top],
        'Score':   [float(row[i]) for i in top],
        'Status':  ['Novel' if A[di,i]==0 else 'Known' for i in top]
    })

case_diseases = [
    'Asthma', 'COPD', 'Type 2 diabetes',
    'Obesity', 'Inflammatory bowel disease(IBD)']

case_results = {}
for d in case_diseases:
    df = get_top_k(d, k=10)
    if df is not None:
        case_results[d] = df
        short = d[:30] + '...' if len(d) > 30 else d
        print(f'\n── Top 10: {short} ──')
        print(df.to_string(index=False))

In [ ]:
# ── CELL 19: Top-20 Microbe Contributors ─────────────────
masked = final_np.copy()
masked[A == 1] = np.nan
mean_score = np.nanmean(masked, axis=0)
top20_idx  = np.argsort(mean_score)[::-1][:20]
top20_df   = pd.DataFrame({
    'Rank':        range(1, 21),
    'Microbe':     [microbes[i] for i in top20_idx],
    'Mean Score':  [float(mean_score[i]) for i in top20_idx],
    'Known Assoc': [int(A[:, i].sum()) for i in top20_idx]
})
print('TOP 20 MICROBE CONTRIBUTORS')
print(top20_df.to_string(index=False))

In [ ]:
# ── CELL 20: Plot — Case Study Bar Charts ────────────────
n_cases = len(case_results)
fig, axes = plt.subplots(1, n_cases, figsize=(5*n_cases, 5.5))
if n_cases == 1: axes = [axes]

for ax, (disease, df) in zip(axes, case_results.items()):
    df10 = df.head(10)
    bar_colors = [C[1] if s=='Known' else C[0]
                  for s in df10['Status'][::-1]]
    ax.barh(df10['Microbe'][::-1], df10['Score'][::-1],
            color=bar_colors, edgecolor='white')
    ax.set_xlabel('Score', fontsize=9)
    title = disease[:25]+'...' if len(disease)>25 else disease
    ax.set_title(title, fontsize=9, fontweight='bold')
    ax.set_xlim(0, 1.05)
    ax.tick_params(axis='y', labelsize=7)

from matplotlib.patches import Patch
fig.legend(handles=[Patch(color=C[1], label='Known'),
                    Patch(color=C[0], label='Novel Prediction')],
           loc='lower center', ncol=2, fontsize=10)
plt.suptitle('Top 10 Novel Microbe Predictions — SGAT-CLMDA',
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('plot_case_studies.png', dpi=200, bbox_inches='tight')
plt.show()

In [ ]:
# ── CELL 21: Plot — Top-20 Contributors ──────────────────
fig, ax = plt.subplots(figsize=(9, 7))
df_plot = top20_df.head(20)
sc = ax.scatter(
    df_plot['Mean Score'], range(len(df_plot)),
    c=df_plot['Known Assoc'], cmap='YlOrRd',
    s=130, zorder=3, vmin=0, vmax=6)
ax.hlines(range(len(df_plot)), 0, df_plot['Mean Score'],
          linewidth=1.5, alpha=0.35, color='grey')
ax.set_yticks(range(len(df_plot)))
ax.set_yticklabels([f"{r['Rank']}. {r['Microbe'][:26]}"
                    for _, r in df_plot.iterrows()], fontsize=8)
plt.colorbar(sc, ax=ax, label='Known Associations in HMDAD')
ax.set_xlabel('Mean Novel Association Score')
ax.set_title('Top 20 Microbe Contributors\n(mean prediction score across all 39 diseases)',
             fontweight='bold')
plt.tight_layout()
plt.savefig('plot_top_contributors.png', dpi=200, bbox_inches='tight')
plt.show()

In [ ]:
# ── CELL 22: Plot — Score Distribution ───────────────────
pos_scores = final_np[pos_d_all, pos_m_all]
np.random.seed(42)
samp = np.random.choice(len(neg_d_all), 3000, replace=False)
neg_scores_s = final_np[neg_d_all[samp], neg_m_all[samp]]

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

ax = axes[0]
ax.hist(neg_scores_s, bins=50, alpha=0.65, color=C[0],
        label=f'Negative (n=3000)\nmean={neg_scores_s.mean():.3f}',
        density=True)
ax.hist(pos_scores,   bins=40, alpha=0.75, color=C[1],
        label=f'Positive (n={len(pos_scores)})\nmean={pos_scores.mean():.3f}',
        density=True)
ax.axvline(0.5, color='black', linestyle='--', lw=1.2, label='Score=0.5')
ax.set_xlabel('Prediction Score'); ax.set_ylabel('Density')
ax.set_title('Score Distribution: Positive vs Negative', fontweight='bold')
ax.legend(fontsize=8)

ax2 = axes[1]
ax2.boxplot([neg_scores_s, pos_scores], labels=['Negative', 'Positive'],
            patch_artist=True,
            boxprops=dict(facecolor='white'),
            medianprops=dict(color='black', lw=2))
for i, (data, col) in enumerate(zip([neg_scores_s, pos_scores], [C[0], C[1]]), 1):
    ax2.scatter([i]*len(data[:200]), data[:200],
                alpha=0.15, s=6, color=col, zorder=3)
ax2.set_ylabel('Score')
ax2.set_title('Score Box Plot', fontweight='bold')

plt.suptitle('Prediction Score Separation Quality',
             fontsize=12, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('plot_score_dist.png', dpi=200, bbox_inches='tight')
plt.show()
print(f'Positive: mean={pos_scores.mean():.4f} std={pos_scores.std():.4f}')
print(f'Negative: mean={neg_scores_s.mean():.4f} std={neg_scores_s.std():.4f}')

In [ ]:
# ── CELL 23: Plot — UMAP Embeddings ──────────────────────
import umap
final_model.eval()
with torch.no_grad():
    _, zd_emb, zm_emb, *_ = final_model(
        X_full, ei_full, FD_f, FM_f, ei_d_f, ei_m_f)
zd_np = zd_emb.cpu().numpy()
zm_np = zm_emb.cpu().numpy()
all_emb = np.vstack([zd_np, zm_np])
reducer = umap.UMAP(n_components=2, random_state=SEED, n_neighbors=15)
emb_2d  = reducer.fit_transform(all_emb)

fig, ax = plt.subplots(figsize=(9, 7))
ax.scatter(emb_2d[nd:,0], emb_2d[nd:,1],
           c=C[0], s=14, alpha=0.45, label='Microbe', zorder=2)
ax.scatter(emb_2d[:nd,0], emb_2d[:nd,1],
           c=C[1], s=160, alpha=0.9, edgecolors='w',
           label='Disease', zorder=3)
for i, d in enumerate(diseases):
    ax.annotate(d[:13], (emb_2d[i,0], emb_2d[i,1]),
                fontsize=5, ha='center',
                xytext=(2,2), textcoords='offset points')
ax.set_title('UMAP Projection of Learned Embeddings — SGAT-CLMDA',
             fontsize=13, fontweight='bold')
ax.legend(fontsize=10)
ax.set_xlabel('UMAP-1'); ax.set_ylabel('UMAP-2')
plt.tight_layout()
plt.savefig('plot_umap.png', dpi=200, bbox_inches='tight')
plt.show()

In [ ]:
# ── CELL 24: Summary Dashboard (all key results) ─────────
fig = plt.figure(figsize=(20, 14))
gs  = gridspec.GridSpec(3, 4, figure=fig, hspace=0.50, wspace=0.38)

# A: ROC
ax1 = fig.add_subplot(gs[0, 0])
ax1.plot(mean_fpr, mean_tpr, lw=2.5, color=C[0],
         label=f'AUC={np.mean(all_auc):.4f}±{np.std(all_auc):.4f}')
ax1.fill_between(mean_fpr, mean_tpr-std_tpr, mean_tpr+std_tpr,
                 alpha=0.12, color=C[0])
ax1.plot([0,1],[0,1],'--', color='grey', lw=1)
ax1.set_xlabel('FPR'); ax1.set_ylabel('TPR')
ax1.set_title('(A) ROC Curve', fontweight='bold')
ax1.legend(fontsize=7)

# B: AUC per fold
ax2 = fig.add_subplot(gs[0, 1])
ax2.bar(range(1, K_FOLDS+1), all_auc, color=C[:K_FOLDS], edgecolor='white')
ax2.axhline(np.mean(all_auc), color='black', lw=1.5, linestyle='--',
            label=f'Mean={np.mean(all_auc):.4f}')
ax2.set_xlabel('Fold'); ax2.set_ylabel('AUC')
ax2.set_ylim(0.85, 1.0)
ax2.set_title('(B) AUC per Fold', fontweight='bold')
ax2.legend(fontsize=7)

# C: Tau evolution
ax3 = fig.add_subplot(gs[0, 2])
for i, th in enumerate(fold_tau_histories):
    ax3.plot(th, lw=1.2, alpha=0.7, color=C[i])
ax3.axhline(0.1, color='grey', linestyle=':', lw=1, label='Init τ=0.1')
ax3.set_xlabel('Epoch'); ax3.set_ylabel('τ')
ax3.set_title('(C) Adaptive τ', fontweight='bold')
ax3.legend(fontsize=7)

# D: Leakage gap
ax4 = fig.add_subplot(gs[0, 3])
ax4.bar(['Honest', 'Leaky'], [honest_mean, leaky_mean],
        color=[C[0], C[1]], edgecolor='white', width=0.45)
ax4.set_ylabel('AUC'); ax4.set_ylim(0.82, 1.0)
ax4.set_title(f'(D) Leakage Gap\n+{leaky_mean-honest_mean:.4f}', fontweight='bold')

# E: Ablation
ax5 = fig.add_subplot(gs[1, 0:2])
abl_m = [np.mean(v) for v in ablation_results.values()]
ax5.barh(list(ablation_results.keys())[::-1], abl_m[::-1],
         color=[C[2] if i==0 else C[7]
                for i in range(len(abl_m))][::-1],
         edgecolor='white', height=0.45)
for i, v in enumerate(abl_m[::-1]):
    ax5.text(v+0.001, i, f'{v:.4f}', va='center', fontsize=8)
ax5.set_xlabel('AUC'); ax5.set_xlim(0.82, 1.0)
ax5.set_title('(E) Ablation Study', fontweight='bold')

# F: Training loss
ax6 = fig.add_subplot(gs[1, 2:4])
for i, l in enumerate(fold_train_losses):
    ax6.plot(l, lw=1.0, alpha=0.6, color=C[i], label=f'F{i+1}')
ax6.set_xlabel('Epoch'); ax6.set_ylabel('Loss')
ax6.set_title('(F) Training Loss', fontweight='bold')
ax6.legend(fontsize=7)

# G: Score distribution
ax7 = fig.add_subplot(gs[2, 0:2])
ax7.hist(neg_scores_s, bins=40, alpha=0.6, color=C[0], density=True, label='Negative')
ax7.hist(pos_scores,   bins=30, alpha=0.7, color=C[1], density=True, label='Positive')
ax7.set_xlabel('Score'); ax7.set_ylabel('Density')
ax7.set_title('(G) Score Distribution', fontweight='bold')
ax7.legend(fontsize=7)

# H: Top contributors
ax8 = fig.add_subplot(gs[2, 2:4])
df10 = top20_df.head(10)
ax8.barh([m[:22] for m in df10['Microbe'][::-1]],
         df10['Mean Score'][::-1], color=C[2], edgecolor='white')
ax8.set_xlabel('Mean Score')
ax8.set_title('(H) Top 10 Microbe Contributors', fontweight='bold')
ax8.tick_params(axis='y', labelsize=7)

plt.suptitle('SGAT-CLMDA — Complete Evaluation Dashboard',
             fontsize=16, fontweight='bold', y=1.01)
plt.savefig('plot_dashboard.png', dpi=200, bbox_inches='tight')
plt.show()

In [ ]:
# ── CELL 25: Save All Outputs ─────────────────────────────
import zipfile, os

# Final metrics summary
print('='*55)
print('SGAT-CLMDA — Final Results Summary')
print('='*55)
print(f'5-Fold CV AUC  : {np.mean(all_auc):.4f} ± {np.std(all_auc):.4f}')
print(f'5-Fold CV AUPR : {np.mean(all_aupr):.4f} ± {np.std(all_aupr):.4f}')
print(f'Leakage gap    : +{leaky_mean - honest_mean:.4f} AUC units')
print(f'τ (final mean) : {np.mean(final_taus):.4f} ± {np.std(final_taus):.4f}')
print('\nAblation:')
for k, v in ablation_results.items():
    print(f'  {k:<25}: {np.mean(v):.4f}')
print('='*55)

# Save model
torch.save(final_model.state_dict(), 'sgat_clmda_final.pt')

# Save predictions
rows = []
for di, disease in enumerate(diseases):
    row = final_np[di].copy()
    top20 = np.argsort(row)[::-1][:20]
    for rank, mi in enumerate(top20):
        rows.append({'Disease': disease, 'Rank': rank+1,
                     'Microbe': microbes[mi], 'Score': float(row[mi]),
                     'Known': int(A[di,mi])})
pd.DataFrame(rows).to_csv('all_predictions.csv', index=False)
top20_df.to_csv('top20_contributors.csv', index=False)

# Zip all files
outputs = [
    'plot_roc.png', 'plot_pr.png', 'plot_loss.png',
    'plot_tau_evolution.png', 'plot_leakage_comparison.png',
    'plot_ablation.png', 'plot_case_studies.png',
    'plot_top_contributors.png', 'plot_score_dist.png',
    'plot_umap.png', 'plot_dashboard.png',
    'all_predictions.csv', 'top20_contributors.csv',
    'sgat_clmda_final.pt'
]
with zipfile.ZipFile('SGAT_CLMDA_outputs.zip', 'w') as zf:
    for f in outputs:
        if os.path.exists(f): zf.write(f)
print('Saved SGAT_CLMDA_outputs.zip')

try:
    from google.colab import files
    files.download('SGAT_CLMDA_outputs.zip')
except: print('Download from file panel.')